# Halide Perovskite X-ray + UV-Vis Optimization — Campaign Notebook

Each trial runs the PL quality gate, records absorbance, then acquires X-ray
scattering.  Both UV-Vis metrics (peak position, FWHM, PLQY) and X-ray PDF
Pearson correlation scores contribute to the multi-objective optimization.

## Prerequisites

**Queue-server worker** must be running with `97-xray_uvvis.py` + `98-hack.py` deployed.

**Environment variables** (set in shell or `.env` before starting Jupyter):
```
QSERVER_HTTP_URI             queue-server HTTP endpoint
QSERVER_HTTP_SERVER_API_KEY  API key for the queue server
ZMQ_CONSUMER_ADDR            ZMQ document-dispatcher address
TILED_URI                    Tiled URI for raw/evaluated data
TILED_PROFILE                Tiled profile name (default: xpd)
```

## Usage modes

**Build fresh** (first run of a new campaign): run all cells top-to-bottom.
The config is archived to `configs/halide_beamline_xray_uvvis.json`.

**Reload prior work**: set `RELOAD_CONFIG = True` in the *Mode toggle* cell
and set `RELOAD_CONFIG_PATH` to the JSON you want to resume from.

## Bring-up sequence

| Step | Notebook |
|------|----------|
| 1 | `validate_paths.ipynb` — device & PV connectivity (read-only) |
| 2 | `single_trial_xray_uvvis.ipynb` — one acquisition trial (pumps + QEPro + pe1c) |
| 3 | **this notebook** — full optimization campaign |

## Environment Setup

In [ ]:
import os
import time
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

HTTP_SERVER_URI: str = os.environ.get(
    "QSERVER_HTTP_URI",
    "https://xf28id2-xpd-qs1.nsls2.bnl.gov",
)
HTTP_API_KEY: str = os.environ.get("QSERVER_HTTP_SERVER_API_KEY", "")
ZMQ_CONSUMER_ADDR: str = os.environ.get(
    "ZMQ_CONSUMER_ADDR",
    "ipc:///var/lib/bluesky-zmq-proxy/xpd-ipc-in-ipc-out/out.sock",
)
TILED_URI: str = os.environ.get("TILED_URI", "https://tiled.nsls2.bnl.gov")
TILED_PROFILE: str = os.environ.get("TILED_PROFILE", "xpd")

AGENT_DATA_PATH: str = "data/halide_data_xray_uvvis.csv"

RELOAD_CONFIG: bool = os.environ.get("RELOAD_CONFIG", "0").strip() in ("1", "true", "yes")
RELOAD_CONFIG_PATH: str = os.environ.get(
    "RELOAD_CONFIG_PATH", "configs/halide_beamline_xray_uvvis.json"
)

print("HTTP_SERVER_URI  :", HTTP_SERVER_URI)
print("ZMQ_CONSUMER_ADDR:", ZMQ_CONSUMER_ADDR)
print("TILED_URI        :", TILED_URI)
print("Mode             :", "RELOAD" if RELOAD_CONFIG else "BUILD FRESH")

## Reference data paths

In [ ]:
_HERE = Path(".").resolve()
REF_DATA = _HERE / "ref_data"

# Verify all phase files exist before building the config
_phase_files = [
    "CsPbBr3_Pnma_mp-567681.gr",   "CsPbBr3_Pnma_mp-567681.cif",
    "CsPbCl3_Pm-3m_mp-23037.gr",   "CsPbCl3_Pm-3m_mp-23037.cif",
    "CsPbI3_Pnma_mp-540839.gr",    "CsPbI3_Pnma_mp-540839.cif",
    "CsBr_Pm-3m_mp-22906.gr",      "CsBr_Pm-3m_mp-22906.cif",
    "Cs4PbBr6_R-3c_mp-23436.gr",   "Cs4PbBr6_R-3c_mp-23436.cif",
]
for _f in _phase_files:
    p = REF_DATA / _f
    status = "✓" if p.exists() else "✗ MISSING"
    print(f"  {status}  {p.name}")

---
## Build or reload the BuildAgent

**Build fresh** (`RELOAD_CONFIG = False`): configure the full experiment below.

**Reload** (`RELOAD_CONFIG = True`): skip to *Build agent* — the agent is
reconstructed from the archived JSON.

In [ ]:
from xpd_tools.optimization.agent import BuildAgent
from xpd_tools.optimization.campaign import (
    assert_plan_registered,
    load_campaign_config,
    save_campaign_config,
)

if RELOAD_CONFIG:
    build_agent = load_campaign_config(RELOAD_CONFIG_PATH)
    print(f"Reloaded config from {RELOAD_CONFIG_PATH}")
    print("  evaluation_method :", build_agent.evaluation_method)
    print("  DOFs              :", [p.id for p in build_agent.pumps])
    print("  run.iterations    :", build_agent.run_settings.iterations)
else:
    print("Building fresh config — run the cells below.")

### UV-Vis objectives

*(Skip this cell on reload.)*

In [ ]:
if not RELOAD_CONFIG:
    from xpd_tools.optimization.helpers.qepro import PlqyReference, SpectraFitSettings

    build_agent = BuildAgent(
        queue_server=True,
        agent_data_path=AGENT_DATA_PATH,
        http_server_uri=HTTP_SERVER_URI,
        http_api_key=HTTP_API_KEY,
        zmq_consumer_address=ZMQ_CONSUMER_ADDR,
        tiled_profile=TILED_PROFILE,
        evaluation_method="xray-uvvis",
    )

    build_agent.set_metadata({
        "beamline": "28id2",
        "tags": ["qserver", "bluesky", "xray", "uvvis"],
        "comment": "Halide Perovskite X-ray + UV-Vis Optimization",
    })

    PEAK_TARGET: float = 520   # nm
    PEAK_TOLERANCE: float = 5  # nm

    build_agent.set_uvvis_objectives(
        peak_target=PEAK_TARGET,
        peak_tolerance=PEAK_TOLERANCE,
        fwhm_role="objective",
        plqy_role="objective",
        peak_distance_role="objective",
        max_peak_distance_nm=PEAK_TOLERANCE * 10,
        min_plqy=1e-3,
        max_fwhm_nm=60,
        max_retries=10,
        retry_delay=2.0,
        use_good_bad=True,
        good_target=3,
        max_bad=3,
        num_abs=10,
        num_flu=10,
        fit_settings=SpectraFitSettings(
            pl_screen_key_height=200,
            pl_screen_peak_height=50,
            pl_screen_peak_distance=100,
            pl_percent_range=(40, 100),
            pl_wavelength_range=(400, 800),
            pl_fit_maxfev=100000,
            pl_fit_r2_window_sigma=3,
            absorbance_percent_range=(10, 70),
            absorbance_wavelength_range=(210, 700),
        ),
        plqy=PlqyReference(
            excitation_wavelength_nm=365,
            absorbance=0.361,
            pl_integral=952628,
            refractive_index=1.337,
            plqy=0.546,
            solvent_refractive_index=1.506,
        ),
    )
    print("UV-Vis objectives set.")

### X-ray objectives

Pearson correlation against five reference phases.
CsPbBr3 is the **objective** (maximize); the remaining four are **constraints**.

*(Skip this cell on reload.)*

In [ ]:
if not RELOAD_CONFIG:
    from xpd_tools.optimization.helpers.phases import Phase

    build_agent.set_xray_objectives(
        exposure=6.0,
        frame_acq_time=1.5,
        no_dark=False,
        stream_name="scattering",
        screening="screen_and_record",
        use_good_bad=True,
        good_target=2,
        max_bad=3,
        num_abs=16,
        num_flu=16,
        objective_function="pearson",
        min_radius=2.0,
        max_radius=20.0,
        max_retries=10,
        retry_delay=2.0,
        phases=[
            Phase(
                name="CsPbBr3",
                gr=str(REF_DATA / "CsPbBr3_Pnma_mp-567681.gr"),
                cif=str(REF_DATA / "CsPbBr3_Pnma_mp-567681.cif"),
                role="objective",
            ),
            Phase(
                name="CsPbCl3",
                gr=str(REF_DATA / "CsPbCl3_Pm-3m_mp-23037.gr"),
                cif=str(REF_DATA / "CsPbCl3_Pm-3m_mp-23037.cif"),
                role="constraint",
                bound=0.3,
                minimize=False,
            ),
            Phase(
                name="CsPbI3",
                gr=str(REF_DATA / "CsPbI3_Pnma_mp-540839.gr"),
                cif=str(REF_DATA / "CsPbI3_Pnma_mp-540839.cif"),
                role="constraint",
                bound=0.1,
                minimize=True,
            ),
            Phase(
                name="CsBr",
                gr=str(REF_DATA / "CsBr_Pm-3m_mp-22906.gr"),
                cif=str(REF_DATA / "CsBr_Pm-3m_mp-22906.cif"),
                role="constraint",
                bound=0.3,
                minimize=True,
            ),
            Phase(
                name="Cs4PbBr6",
                gr=str(REF_DATA / "Cs4PbBr6_R-3c_mp-23436.gr"),
                cif=str(REF_DATA / "Cs4PbBr6_R-3c_mp-23436.cif"),
                role="constraint",
                bound=0.3,
                minimize=True,
            ),
        ],
    )
    print("X-ray objectives set.")

### DOFs and experiment

*(Skip this cell on reload.)*

In [ ]:
if not RELOAD_CONFIG:
    from xpd_tools.optimization.helpers.dofs import Pump
    from xpd_tools.optimization.plans import DilutionStage, FlowSource, WashCycle

    _dds2_p1 = "CsPb"
    _dds2_p2 = "TOABr"
    _dds3_p1 = "Cl"
    _dds3_p2 = "I"

    build_agent.set_dofs([
        Pump(name=_dds2_p1, id="dds2_p1", bounds=(0, 100)),
        Pump(name=_dds2_p2, id="dds2_p2", bounds=(0, 100)),
        Pump(name=_dds3_p1, id="dds3_p1", bounds=(0, 100)),
        Pump(name=_dds3_p2, id="dds3_p2", bounds=(0, 100)),
    ])

    build_agent.experiment(
        sources=[
            FlowSource(
                dof="infusion_rate_CsPb",  pump="dds2_p1",
                precursor=_dds2_p1, sample_label="Cs-rich 33mM", loaded_ml=30.0,
            ),
            FlowSource(
                dof="infusion_rate_TOABr", pump="dds2_p2",
                precursor=_dds2_p2, sample_label="TOABr",        loaded_ml=30.0,
            ),
            FlowSource(
                dof="infusion_rate_Cl",    pump="dds3_p1",
                precursor=_dds3_p1, sample_label="Pb-rich 33mM", loaded_ml=30.0,
            ),
            FlowSource(
                dof="infusion_rate_I",     pump="dds3_p2",
                precursor=_dds3_p2, sample_label="I",             loaded_ml=30.0,
            ),
        ],
        dilutions=[
            DilutionStage(
                pump="dds1_p1", ratio=1.0, position="before_equilibrium",
                syringe_ml=20, material="plastic_BD", target_ml=20, loaded_ml=20.0,
            ),
            DilutionStage(
                pump="ultra2", ratio=1.0, position="after_equilibrium",
                syringe_ml=100, material="steel", target_ml=100,
                loaded_ml=100.0, wait_sec=30,
            ),
        ],
        wash_cycles=[
            WashCycle(pump="ultra1", loaded_ml=30.0),
        ],
        rate_unit="ul/min",
        mixer_lengths_cm=(30.0,),
        residence_time_ratio=1.0,
        tubing_id_mm=1.016,
    )

    build_agent.set_run(
        iterations=60,
        extra_initialization_trials=5,
        generation_strategy={
            "initialize_with_center": False,
            "min_observed_initialization_trials": 2,
            "allow_exceeding_initialization_budget": True,
        },
    )

    cfg_path = save_campaign_config(build_agent, "halide_beamline_xray_uvvis")
    print(f"Config archived → {cfg_path}")

---
## Build the QueueserverAgent and verify plan registration

In [ ]:
run_agent = build_agent.build()

print("QueueserverAgent built.")
print("  acquisition_plan :", run_agent.acquisition_plan)
print("  actuators        :", list(run_agent.actuators))

assert_plan_registered(run_agent, plan_name="xray_uvvis_acquire")

---
## Seed historical data

In [ ]:
n_historical = build_agent.configure_generation_strategy(run_agent)
print(f"{n_historical} historical trials ingested into the generation strategy.")

---
## Run the optimization campaign

In [ ]:
campaign_future = run_agent.run(
    iterations=build_agent.run_settings.iterations,
    n_points=build_agent.run_settings.n_points,
    checkpoint_interval=10,
)

print(f"Campaign started — {build_agent.run_settings.iterations} iterations.")
print("Watching progress (interrupt the kernel or run_agent.stop() to halt early)...\n")

while not campaign_future.done():
    print(
        f"  iteration {run_agent.current_iteration}"
        f" / {build_agent.run_settings.iterations}"
    )
    time.sleep(30)

campaign_result = campaign_future.result()
print("\nCampaign complete.")
print(campaign_result)